<a href="https://colab.research.google.com/github/kshitijtandon/fisabio-gapseq-workshop/blob/main/notebooks/FISABIO_gapseq_workshop.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FISABIO gapseq workshop

This is the live demo of how to develop a genome-scale metabolic model of a bacterial genome or Metagenome-assembled-genome using [gapseq](https://link.springer.com/article/10.1186/s13059-021-02295-1$0).

I have created a [GitHub]() repo for ease of analysis. As generating a GEM can take several hours. So, you will have all the files ready for you view while we go through the steps.




In [1]:
# Clone the workshop repository

!git clone https://github.com/kshitijtandon/fisabio-gapseq-workshop.git

%cd fisabio-gapseq-workshop

Cloning into 'fisabio-gapseq-workshop'...
remote: Enumerating objects: 33, done.
remote: Counting objects: 100% (33/33), done.
remote: Compressing objects: 100% (26/26), done.
remote: Total 33 (delta 5), reused 29 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (33/33), 5.62 MiB | 13.72 MiB/s, done.
Resolving deltas: 100% (5/5), done.
/content/fisabio-gapseq-workshop


Basic workflow to create a GEM is as follows

## gapseq workflow

| Step | Command | Output |
|------|---------|--------|
| 1 | `gapseq find` | Reaction predictions, pathway predictions |
| 2 | `gapseq find-transport` | Transporter predictions |
| 3 | `gapseq draft` | Draft metabolic model |
| 4 | `gapseq fill` | Gap-filled metabolic model |

In [2]:
%%bash

echo "Repository contents:"

ls -lh

echo ""

echo "Data files:"

ls -lh data

Repository contents:
total 16K
drwxr-xr-x 2 root root 4.0K Jun 25 03:17 data
drwxr-xr-x 2 root root 4.0K Jun 25 03:17 notebooks
-rw-r--r-- 1 root root  801 Jun 25 03:17 README.md
drwxr-xr-x 2 root root 4.0K Jun 25 03:17 scripts

Data files:
total 35M
-rw-r--r-- 1 root root 173K Jun 25 03:17 ecoli_k12-all-Pathways.tbl
-rw-r--r-- 1 root root 8.6M Jun 25 03:17 ecoli_k12-all-Reactions.tbl
-rw-r--r-- 1 root root 801K Jun 25 03:17 ecoli_k12-draft.RDS
-rw-r--r-- 1 root root  11M Jun 25 03:17 ecoli_k12-draft.xml
-rw-r--r-- 1 root root 881K Jun 25 03:17 ecoli_k12.faa.gz
-rw-r--r-- 1 root root 1.4M Jun 25 03:17 ecoli_k12_genome.fna.gz
-rw-r--r-- 1 root root 810K Jun 25 03:17 ecoli_k12.RDS
-rw-r--r-- 1 root root 134K Jun 25 03:17 ecoli_k12-rxnWeights.RDS
-rw-r--r-- 1 root root 304K Jun 25 03:17 ecoli_k12-rxnXgenes.RDS
-rw-r--r-- 1 root root 642K Jun 25 03:17 ecoli_k12-Transporter.tbl
-rw-r--r-- 1 root root  11M Jun 25 03:17 ecoli_k12.xml
-rw-r--r-- 1 root root  440 Jun 25 03:17 MM_glu.csv


# Genome-scale metabolic modelling workflow

🧬 Protein FASTA

⬇️

🔍 **find**

• Predict reactions

• Predict pathways

⬇️

🚪 **find-transporters**

• Predict transporters

⬇️

🏗️ **develop a draft**

• Build draft metabolic model

⬇️

🩹 **filling missing elements**

• Fill missing reactions

⬇️

🧫 **Genome-scale metabolic model**

# gapseq workflow

| Step | Command | Output |
|------|---------|--------|
| 1 | `gapseq find` | Reaction predictions, pathway predictions |
| 2 | `gapseq find-transport` | Transporter predictions |
| 3 | `gapseq draft` | Draft metabolic model |
| 4 | `gapseq fill` | Gap-filled metabolic model |

## Step 1: Input protein sequences

gapseq starts from protein sequences predicted from a genome assembly.

For this workshop we use:

**Escherichia coli K-12**

In [7]:
import gzip

n_proteins = 0
protein_lengths = []

with gzip.open("data/ecoli_k12.faa.gz", "rt") as f:
    seq = ""
    for line in f:
        if line.startswith(">"):
            if seq:
                protein_lengths.append(len(seq))
            n_proteins += 1
            seq = ""
        else:
            seq += line.strip()
    if seq:
        protein_lengths.append(len(seq))

print(f"Proteins: {n_proteins:,}")
print(f"Average protein length: {sum(protein_lengths)/len(protein_lengths):.1f} aa")
print(f"Longest protein: {max(protein_lengths):,} aa")

Proteins: 4,298
Average protein length: 309.4 aa
Longest protein: 2,358 aa


## Step 2: Predict reactions and pathways

The first major gapseq step is:

```bash
gapseq find -p all ecoli_k12.faa.gz
```

This command searches the predicted protein sequences against curated metabolic databases and maps genes to metabolic functions.

For the workshop, we have already run this step because it can take some time to complete. The command generates several output files that will be used in the subsequent steps of the workflow.

| **Output file** | **Description** |
|-----------------|-----------------|
| `ecoli_k12-all-Reactions.tbl` | Predicted metabolic reactions identified from the protein sequences. |
| `ecoli_k12-all-Pathways.tbl` | Predicted metabolic pathways reconstructed from the identified reactions. |
| `ecoli_k12-rxnWeights.RDS` | Confidence scores assigned to each predicted reaction. These scores are used during the gap-filling step to prioritise high-confidence reactions. |
| `ecoli_k12-rxnXgenes.RDS` | Mapping between predicted reactions and the genes supporting each reaction. |

In [10]:
import pandas as pd

reactions = pd.read_csv(
    "data/ecoli_k12-all-Reactions.tbl",
    sep="\t",
    comment="#",
    engine="python"
)

pathways = pd.read_csv(
    "data/ecoli_k12-all-Pathways.tbl",
    sep="\t",
    comment="#",
    engine="python"
)

Predicted reactions table: 31,988 rows × 19 columns
Predicted pathways table: 1,921 rows × 8 columns


Let's take a look at some predicted reactions and pathways

In [13]:
display(reactions.head(5))

,rxn,name,ec,bihit,qseqid,pident,evalue,bitscore,qcovs,stitle,sstart,send,pathway,status,pathway.status,dbhit,complex,exception,complex.status
0,DHLAXANAU-RXN,"1,2-dichloroethane dehalogenase",3.8.1.5,NaN,UniRef90_P9WMS1,25.085,1.130000e-11,60.8,96.0,"NP_414883.5 2-hydroxy-6-ketonona-2,4-dienedioa...",5.0,283.0,|12DICHLORETHDEG-PWY|,bad_blast,NaN,rxn03596 rxn03668 rxn03669 rxn03670 rxn03671 r...,NaN,0,NaN
1,MOXXANAU-RXN,2-chloroethanol dehydrogenase,1.1.2.7,NaN,UniRef90_P38539,26.614,1.230000e-40,154.0,89.0,NP_414666.1 quinoprotein glucose dehydrogenase...,168.0,768.0,|12DICHLORETHDEG-PWY|,bad_blast,NaN,rxn00847 rxn01796 rxn12745 rxn14207 rxn15989 r...,Subunit 1,0,NaN
2,MOXXANAU-RXN,2-chloroethanol dehydrogenase,1.1.2.7,NaN,UniRef90_Q09053,41.176,1.200000e+00,20.8,94.0,NP_415294.1 putative kinase inhibitor [Escheri...,25.0,41.0,|12DICHLORETHDEG-PWY|,bad_blast,NaN,rxn00847 rxn01796 rxn12745 rxn14207 rxn15989 r...,Subunit 2,0,NaN
3,ALDXANAU-RXN,chloroacetaldehyde dehydrogenase,1.2.1.4,NaN,UniRef90_P37685,100.000,0.000000e+00,1065.0,100.0,NP_418045.4 aldehyde dehydrogenase B [Escheric...,1.0,512.0,|12DICHLORETHDEG-PWY|,good_blast,NaN,rxn00507 rxn03455 rxn05880 rxn14071 rxn19116,NaN,0,1.0
4,ALDXANAU-RXN,chloroacetaldehyde dehydrogenase,1.2.1.4,NaN,UniRef90_P37685,40.333,1.440000e-110,334.0,93.0,NP_415816.1 gamma-glutamyl-gamma-aminobutyrald...,23.0,492.0,|12DICHLORETHDEG-PWY|,good_blast,NaN,rxn00507 rxn03455 rxn05880 rxn14071 rxn19116,NaN,0,1.0


### Reaction predictions

Each row represents a predicted metabolic reaction supported by protein-level evidence from the genome.

These reaction predictions are one of the main inputs used to construct the draft metabolic model.

In [15]:
display(pathways.head(5))

,ID,Name,Prediction,Completeness,VagueReactions,KeyReactions,KeyReactionsFound,ReactionsFound
0,|12DICHLORETHDEG-PWY|,"1,2-dichloroethane degradation",False,25,0,1,0,ALDXANAU-RXN
1,|14DICHLORBENZDEG-PWY|,"1,4-dichlorobenzene degradation",False,42,0,2,1,CHLOROBENDDH-RXN DIENELACHYDRO-RXN MALEYLACETA...
2,|1CMET2-PWY|,folate transformations III (E. coli),True,100,0,0,0,DIHYDROFOLATEREDUCT-RXN 5-FORMYL-THF-CYCLO-LIG...
3,|2AMINOBENZDEG-PWY|,anthranilate degradation III (anaerobic),False,0,0,0,0,NaN
4,|2ASDEG-PWY|,orthanilate degradation,False,25,1,0,0,NaN


### Pathway predictions

Pathway predictions summarise the metabolic capabilities inferred from the collection of predicted reactions.

This helps move from individual enzyme/reaction evidence to broader biological functions.

## How many complete pathways can you find from top 5 pathway hits?

#How many reactions and pathways were predicted?